In [1]:
# Cell 1: Imports (No changes needed)
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from collections import deque
import sys
import time # Added for timing local search if needed


In [2]:
# Cell 2: HybridPowerFlowOptimizer Class (Net Injection Model)

class HybridPowerFlowOptimizer:
    """
    Optimizes power flow using a hybrid metaheuristic approach, handling
    buses with fixed load and controllable generation based on user specification.
    Allows zero-cost load shedding ONLY for buses designated as pure loads.
    """

    # Added gen_indices, load_indices, fixed_load to constructor
    def __init__(self, A_matrix, line_limits, gen_costs_full, gen_limits_min_full, gen_limits_max_full,
                 initial_B_net, fixed_load_full, # Pass initial NET injection and FIXED load
                 gen_indices, load_indices, memory_size=100):
        self.A = A_matrix
        self.line_limits = np.array(line_limits, dtype=np.float64)
        self.num_lines = A_matrix.shape[0]
        self.num_buses = A_matrix.shape[1]
        self.tolerance = 1e-6

        # --- Store Initial Net Injection, Fixed Load, and Indices ---
        self.initial_B_net = initial_B_net.copy().flatten() # Store initial NET injection
        self.fixed_load = fixed_load_full.copy().flatten() # Store fixed load (Pl >= 0)
        if np.any(self.fixed_load < 0):
             print("Warning: Fixed loads should be non-negative (representing demand). Clamping negative loads to zero.")
             self.fixed_load = np.maximum(0, self.fixed_load)

        self.gen_indices = np.array(sorted(gen_indices), dtype=int)
        self.load_indices = np.array(sorted(load_indices), dtype=int)

        # Validation
        all_indices = np.concatenate((self.gen_indices, self.load_indices))
        if len(np.unique(all_indices)) != self.num_buses or len(all_indices) != self.num_buses:
             raise ValueError("Generator and Load indices do not form a complete, non-overlapping set of all buses.")
        # Note: Allowing zero generators might be valid if only load shedding is possible

        print(f"Using specified {len(self.gen_indices)} generator buses (indices: {self.gen_indices})")
        print(f"Using specified {len(self.load_indices)} load-only buses (indices: {self.load_indices})")

        # --- Store Generator-Specific Data (Pg limits and costs) ---
        num_gens = len(self.gen_indices)
        self.gen_costs_only = np.zeros(num_gens)
        self.gen_limits_Pg_min_only = np.zeros(num_gens) # Store Pg limits
        self.gen_limits_Pg_max_only = np.zeros(num_gens) # Store Pg limits

        if num_gens > 0:
             flat_gc = np.array(gen_costs_full).flatten()
             flat_gmin = np.array(gen_limits_min_full).flatten()
             flat_gmax = np.array(gen_limits_max_full).flatten()
             if len(flat_gc)!=self.num_buses or len(flat_gmin)!=self.num_buses or len(flat_gmax)!=self.num_buses:
                 raise ValueError(f"Generator cost/limit array length mismatch (Expected {self.num_buses})")

             self.gen_costs_only = flat_gc[self.gen_indices]
             self.gen_limits_Pg_min_only = flat_gmin[self.gen_indices]
             self.gen_limits_Pg_max_only = flat_gmax[self.gen_indices]

             if np.any(self.gen_limits_Pg_max_only < self.gen_limits_Pg_min_only):
                 raise ValueError("Generator Max Pg limit cannot be less than Min Pg limit.")

        # --- Calculate and Store NET Injection Limits ---
        # For Generators: B_net = Pg - Pl_fixed => Pg_min - Pl <= B_net <= Pg_max - Pl
        self.gen_limits_Bnet_min_only = np.zeros(num_gens)
        self.gen_limits_Bnet_max_only = np.zeros(num_gens)
        if num_gens > 0:
             fixed_load_at_gens = self.fixed_load[self.gen_indices]
             self.gen_limits_Bnet_min_only = self.gen_limits_Pg_min_only - fixed_load_at_gens
             self.gen_limits_Bnet_max_only = self.gen_limits_Pg_max_only - fixed_load_at_gens

        # For Load-Only buses: B_net = 0 - Pl => -Pl_fixed <= B_net <= 0 (Shedding allowed)
        num_loads = len(self.load_indices)
        self.load_limits_Bnet_min_only = np.zeros(num_loads) # Min net injection = -Pl_fixed
        self.load_limits_Bnet_max_only = np.zeros(num_loads) # Max net injection = 0 (fully shed)
        if num_loads > 0:
             fixed_load_at_loads = self.fixed_load[self.load_indices]
             self.load_limits_Bnet_min_only = 0.0 - fixed_load_at_loads # B can go down to -Pl
             # self.load_limits_Bnet_max_only is already zeros

        # --- Adjust Initial NET Injection State if Necessary ---
        needs_adjust = False
        # 1. Clamp initial NET injection at generator buses if implied Pg is outside limits
        if num_gens > 0:
            initial_Bnet_at_gens = self.initial_B_net[self.gen_indices]
            # Clip B_net based on calculated net limits for generators
            clipped_Bnet_gens = np.clip(initial_Bnet_at_gens,
                                        self.gen_limits_Bnet_min_only,
                                        self.gen_limits_Bnet_max_only)
            if np.any(np.abs(initial_Bnet_at_gens - clipped_Bnet_gens) > self.tolerance):
                 print("Warning: Initial net injection at generator bus(es) implies Pg outside limits. Clamping net injection...")
                 self.initial_B_net[self.gen_indices] = clipped_Bnet_gens
                 needs_adjust = True

        # 2. Clamp initial NET injection at load buses (should be <= 0)
        if num_loads > 0:
             initial_Bnet_at_loads = self.initial_B_net[self.load_indices]
             # Clip B_net based on calculated net limits for loads (-Pl <= B <= 0)
             clipped_Bnet_loads = np.clip(initial_Bnet_at_loads,
                                          self.load_limits_Bnet_min_only,
                                          self.load_limits_Bnet_max_only)
             if np.any(np.abs(initial_Bnet_at_loads - clipped_Bnet_loads) > self.tolerance):
                 print("Warning: Initial net injection at load bus(es) outside limits [-Pl, 0]. Clamping...")
                 self.initial_B_net[self.load_indices] = clipped_Bnet_loads
                 needs_adjust = True


        # 3. Balance initial NET injections (sum should be close to zero) by adjusting GENERATORS
        required_total_injection = 0.0
        current_total_injection = np.sum(self.initial_B_net)
        difference_init = required_total_injection - current_total_injection

        if abs(difference_init) > self.tolerance * self.num_buses:
            print(f"Warning: Initial NET injections sum to {current_total_injection:.4f} (≠ 0). Adjusting GENERATORS' net injection to balance...")
            if num_gens > 0:
                diff_per_gen_init = difference_init / num_gens
                adjusted_Bnet_gens = self.initial_B_net[self.gen_indices] + diff_per_gen_init
                # Clip adjustment based on *net injection* limits for generators
                clipped_adjusted_Bnet_gens = np.clip(adjusted_Bnet_gens,
                                                     self.gen_limits_Bnet_min_only,
                                                     self.gen_limits_Bnet_max_only)
                actual_adjustment_applied = np.sum(clipped_adjusted_Bnet_gens) - np.sum(self.initial_B_net[self.gen_indices])
                self.initial_B_net[self.gen_indices] = clipped_adjusted_Bnet_gens
                remaining_diff = difference_init - actual_adjustment_applied
                if abs(remaining_diff) > self.tolerance * self.num_buses:
                     print(f"Warning: Could not fully balance initial state due to generator net injection limits. Remaining imbalance: {remaining_diff:.4f}")
                needs_adjust = True
            else:
                print("ERROR: Cannot balance initial state - no generators specified.")
                needs_adjust = True # Mark as adjusted, though problematic

        if needs_adjust:
            print("-> Adjusted initial Net Injection (B) state used for optimization:", np.round(self.initial_B_net, 4))
        # --- End Initial State Handling ---

        self.memory = deque(maxlen=memory_size)
        self.stagnation_threshold = 50
        self.diversification_fraction = 0.2

    # --- Methods _apply_constraints, _calculate_fitness, _is_feasible, etc. ---
    # These now operate on NET INJECTION (B) but use the derived Bnet limits

    def _apply_constraints(self, solution_vector):
        """Applies NET injection limits and enforces power balance."""
        sol_Bnet = solution_vector.copy().flatten() # Working with net injection
        num_gens = len(self.gen_indices)
        num_loads = len(self.load_indices)

        # 1. Apply Load-Only Bus Net Injection Limits (Shedding: -Pl <= B <= 0)
        if num_loads > 0:
            sol_Bnet[self.load_indices] = np.clip(sol_Bnet[self.load_indices],
                                                  self.load_limits_Bnet_min_only,
                                                  self.load_limits_Bnet_max_only)

        # 2. Apply Generator Bus Net Injection Limits (Pg_min - Pl <= B <= Pg_max - Pl)
        if num_gens > 0:
            sol_Bnet[self.gen_indices] = np.clip(sol_Bnet[self.gen_indices],
                                                 self.gen_limits_Bnet_min_only,
                                                 self.gen_limits_Bnet_max_only)

        # 3. Enforce Power Balance (Adjust Generator NET Injections)
        if num_gens > 0:
            # Balance requires sum of NET injections = 0
            required_total_Bnet = 0.0
            current_total_Bnet = np.sum(sol_Bnet)
            difference = required_total_Bnet - current_total_Bnet

            if abs(difference) > self.tolerance * self.num_buses:
                adjustment_per_gen = difference / num_gens
                adjusted_Bnet_gens = sol_Bnet[self.gen_indices] + adjustment_per_gen
                # Re-clip using generator net injection limits
                clipped_adjusted_Bnet_gens = np.clip(adjusted_Bnet_gens,
                                                     self.gen_limits_Bnet_min_only,
                                                     self.gen_limits_Bnet_max_only)
                actual_adjustment_applied = np.sum(clipped_adjusted_Bnet_gens) - np.sum(sol_Bnet[self.gen_indices])
                remaining_diff = difference - actual_adjustment_applied
                sol_Bnet[self.gen_indices] = clipped_adjusted_Bnet_gens
                # Optional: Add warning if remaining_diff is large

        return sol_Bnet.reshape(-1, 1)

    def _calculate_fitness(self, solution_Bnet):
        """Calculates fitness based on NET injection (B)."""
        sol_Bnet_flat = solution_Bnet.flatten()
        if not np.all(np.isfinite(sol_Bnet_flat)): return np.inf
        try:
            C = np.dot(self.A, sol_Bnet_flat); flows = C.flatten()
            if not np.all(np.isfinite(C)): return np.inf
        except ValueError: return np.inf

        penalty_multiplier = 1e10

        # 1. Line Limit Penalty (Based on flows from B_net)
        line_violations = np.maximum(0, np.abs(flows) - (self.line_limits + self.tolerance))
        line_violation_penalty = penalty_multiplier * np.sum(line_violations**2)

        # 2. Generator Net Injection Limit Penalty
        gen_limit_penalty = 0.0
        if len(self.gen_indices) > 0:
            gen_Bnet_values = sol_Bnet_flat[self.gen_indices]
            violations_lower_g = np.maximum(0, self.gen_limits_Bnet_min_only - gen_Bnet_values + self.tolerance)
            violations_upper_g = np.maximum(0, gen_Bnet_values - self.gen_limits_Bnet_max_only - self.tolerance)
            gen_limit_penalty = penalty_multiplier * (np.sum(violations_lower_g**2) + np.sum(violations_upper_g**2))

        # 3. Load-Only Net Injection Limit Penalty
        load_limit_penalty = 0.0
        if len(self.load_indices) > 0:
            load_Bnet_values = sol_Bnet_flat[self.load_indices]
            violations_lower_l = np.maximum(0, self.load_limits_Bnet_min_only - load_Bnet_values + self.tolerance)
            violations_upper_l = np.maximum(0, load_Bnet_values - self.load_limits_Bnet_max_only - self.tolerance)
            load_limit_penalty = penalty_multiplier * (np.sum(violations_lower_l**2) + np.sum(violations_upper_l**2))

        # 4. Power Balance Penalty (Based on sum of B_net)
        balance_violation = abs(np.sum(sol_Bnet_flat))
        balance_penalty = penalty_multiplier * (balance_violation**2) if balance_violation > self.tolerance * self.num_buses else 0.0

        # --- Objective Components ---
        # 5. Generator Deviation Penalty (Based on change in B_net for generators)
        #    Change in B_net = Change in Pg (since Pl is fixed for these buses)
        generator_deviation = self._get_generator_deviation(solution_Bnet) # Uses B_net
        deviation_penalty_component = 1e5 * generator_deviation # Tunable weight

        # 6. Generator Rescheduling Cost (Based on change in B_net for generators)
        gen_rescheduling_cost = self._get_rescheduling_cost(solution_Bnet) # Uses B_net
        gen_cost_weight = 0.1 # Tunable weight

        fitness = (line_violation_penalty + gen_limit_penalty + load_limit_penalty +
                   balance_penalty + deviation_penalty_component + gen_cost_weight * gen_rescheduling_cost)
        return fitness if np.isfinite(fitness) else np.inf

    def _is_feasible(self, solution_Bnet, verbose=False):
        """Checks if a NET injection solution meets all hard constraints."""
        if solution_Bnet is None:
            if verbose: print("DEBUG (_is_feasible): Input solution is None.")
            return False
        sol_Bnet_flat = solution_Bnet.flatten()
        if not np.all(np.isfinite(sol_Bnet_flat)):
            if verbose: print("DEBUG (_is_feasible): Solution contains non-finite values.")
            return False

        # Check Line Limits
        line_ok = False; flows = np.array([])
        try:
            C = np.dot(self.A, sol_Bnet_flat); flows = C.flatten()
            if not np.all(np.isfinite(C)): raise ValueError("Flows NaN/Inf")
            line_ok = np.all(np.abs(flows) <= self.line_limits + self.tolerance)
        except Exception as e: line_ok = False;

        # Check Generator Net Injection Limits
        gen_ok = False
        if len(self.gen_indices) > 0:
             gen_Bnet_values = sol_Bnet_flat[self.gen_indices]
             gen_ok = np.all(gen_Bnet_values >= self.gen_limits_Bnet_min_only - self.tolerance) and \
                      np.all(gen_Bnet_values <= self.gen_limits_Bnet_max_only + self.tolerance)
        else: gen_ok = True # No generators

        # Check Load-Only Net Injection Limits
        load_ok = False
        if len(self.load_indices) > 0:
             load_Bnet_values = sol_Bnet_flat[self.load_indices]
             load_ok = np.all(load_Bnet_values >= self.load_limits_Bnet_min_only - self.tolerance) and \
                       np.all(load_Bnet_values <= self.load_limits_Bnet_max_only + self.tolerance)
        else: load_ok = True # No load-only buses

        # Check Power Balance (Sum of Net Injections)
        bal_ok = abs(np.sum(sol_Bnet_flat)) < self.tolerance * self.num_buses

        feasible = line_ok and gen_ok and load_ok and bal_ok

        # Verbose Output (logic remains similar, checks Bnet limits)
        if verbose or not feasible:
            print(f"--- Feasibility Check {'FAILED' if not feasible else 'PASSED'} ---")
            if not line_ok: print(f"  Line constraints failed."); # Add details if needed
            else: print("  Line constraints met.")
            if not gen_ok: print(f"  Generator net injection limits failed (Buses: {self.gen_indices+1}).")
            else: print("  Generator net injection limits met.")
            if not load_ok: print(f"  Load-only net injection limits failed (Buses: {self.load_indices+1}).")
            else: print("  Load-only net injection limits met.")
            if not bal_ok: print(f"  Power balance check failed, sum(B_net)={np.sum(sol_Bnet_flat):.8f}")
            else: print("  Power balance met.")
            print("-" * 40)
        return feasible

    # --- Helper methods for objectives (operate on B_net) ---
    def _get_generator_deviation(self, solution_Bnet):
        """Calculates sum of absolute changes in B_net for generators."""
        sol_Bnet_flat = solution_Bnet.flatten()
        if len(self.gen_indices) == 0: return 0.0
        try:
            # Compare current B_net with initial B_net for generator indices
            if np.max(self.gen_indices) >= len(sol_Bnet_flat) or np.max(self.gen_indices) >= len(self.initial_B_net): return np.inf
            return np.sum(np.abs(sol_Bnet_flat[self.gen_indices] - self.initial_B_net[self.gen_indices]))
        except IndexError: return np.inf

    def _get_rescheduling_cost(self, solution_Bnet):
        """Calculates cost based on changes in B_net for generators."""
        sol_Bnet_flat = solution_Bnet.flatten()
        if len(self.gen_indices) == 0: return 0.0
        try:
            # Use gen_costs_only (cost per MW change in Pg, which equals change in B_net)
            if len(self.gen_costs_only) != len(self.gen_indices) or \
               np.max(self.gen_indices) >= len(sol_Bnet_flat) or \
               np.max(self.gen_indices) >= len(self.initial_B_net): return np.inf
            # Deviation is change in B_net
            deviation_Bnet = np.abs(sol_Bnet_flat[self.gen_indices] - self.initial_B_net[self.gen_indices])
            cost = np.sum(self.gen_costs_only * deviation_Bnet)
            return cost
        except IndexError: return np.inf

    # --- Random Solution Generation (operates on B_net) ---
    def _generate_random_solution(self):
        """Generates a new random NET injection solution."""
        rand_sol_Bnet = self.initial_B_net.copy()
        n_gens = len(self.gen_indices)
        n_loads = len(self.load_indices)

        # Perturb Generator Net Injection within their Bnet limits
        if n_gens > 0:
            gen_Bnet_range = np.maximum(self.tolerance, self.gen_limits_Bnet_max_only - self.gen_limits_Bnet_min_only)
            perturbation_factor = 0.2
            perturbations_g = (np.random.rand(n_gens) - 0.5) * gen_Bnet_range * perturbation_factor
            rand_sol_Bnet[self.gen_indices] = np.clip(self.initial_B_net[self.gen_indices] + perturbations_g,
                                                      self.gen_limits_Bnet_min_only, self.gen_limits_Bnet_max_only)

        # Perturb Load-Only Net Injection (Allowing shedding towards 0)
        if n_loads > 0:
            # Range is from min Bnet (-Pl) up to max Bnet (0)
            load_Bnet_range = self.load_limits_Bnet_max_only - self.load_limits_Bnet_min_only # Should be Pl (>=0)
            shedding_factor = 0.1 # Max % of range to perturb by
            # Perturbation increases Bnet (reduces load magnitude)
            perturbations_l = np.random.rand(n_loads) * load_Bnet_range * shedding_factor
            rand_sol_Bnet[self.load_indices] = np.clip(self.initial_B_net[self.load_indices] + perturbations_l,
                                                       self.load_limits_Bnet_min_only, self.load_limits_Bnet_max_only)

        return self._apply_constraints(rand_sol_Bnet) # Apply constraints (incl. balancing)

    # --- Local Search (operates on B_net) ---
    def _apply_local_search(self, solution_Bnet, iteration, max_iterations):
        """Applies simple local search heuristic to refine NET injection solution."""
        current_solution_Bnet = solution_Bnet.copy()
        current_fitness = self._calculate_fitness(current_solution_Bnet)
        if not np.isfinite(current_fitness): return solution_Bnet

        progress_ratio = iteration / max_iterations
        step_scale_factor = 0.1 * (1.0 - progress_ratio) + 0.01 * progress_ratio
        num_attempts = min(5, int(np.sqrt(self.num_buses)))

        for _ in range(num_attempts):
            idx_to_perturb = random.randrange(self.num_buses)
            perturbed_solution_Bnet = current_solution_Bnet.copy()
            perturb_range = 1.0 # Default

            # Determine perturbation range based on Bnet limits
            if idx_to_perturb in self.gen_indices:
                gen_idx_local = np.where(self.gen_indices == idx_to_perturb)[0][0]
                op_range = self.gen_limits_Bnet_max_only[gen_idx_local] - self.gen_limits_Bnet_min_only[gen_idx_local]
                perturb_range = max(self.tolerance, op_range) * step_scale_factor
            elif idx_to_perturb in self.load_indices:
                load_idx_local = np.where(self.load_indices == idx_to_perturb)[0][0]
                op_range = self.load_limits_Bnet_max_only[load_idx_local] - self.load_limits_Bnet_min_only[load_idx_local]
                perturb_range = max(self.tolerance, op_range) * step_scale_factor

            perturbation = (random.random() - 0.5) * 2 * perturb_range
            perturbed_solution_Bnet[idx_to_perturb, 0] += perturbation

            refined_solution_Bnet = self._apply_constraints(perturbed_solution_Bnet) # Apply constraints
            refined_fitness = self._calculate_fitness(refined_solution_Bnet)

            if np.isfinite(refined_fitness) and refined_fitness < current_fitness:
                current_solution_Bnet = refined_solution_Bnet
                current_fitness = refined_fitness

        return current_solution_Bnet

    # --- Optimization Loop (operates on B_net) ---
    def optimize(self, B_unused, iterations=200, population_size=30):
        """Main optimization loop using hybrid metaheuristics (operates on B_net)."""
        print(f"Starting Optimization: Pop={population_size}, Iterations={iterations}")
        initial_state_constrained = self._apply_constraints(self.initial_B_net) # Start with adjusted initial B_net
        initial_population = [self._generate_random_solution() for _ in range(population_size)]
        initial_population[0] = initial_state_constrained
        fitness_values = [self._calculate_fitness(sol) for sol in initial_population]
        best_idx = np.argmin(fitness_values)
        best_solution_Bnet = initial_population[best_idx].copy()
        best_fitness = fitness_values[best_idx]

        if not np.isfinite(best_fitness): # Handle case where initial population is bad
             finite_indices = np.where(np.isfinite(fitness_values))[0]
             if len(finite_indices) > 0:
                 best_idx = finite_indices[np.argmin(np.array(fitness_values)[finite_indices])]
                 best_solution_Bnet = initial_population[best_idx].copy(); best_fitness = fitness_values[best_idx]
                 print(f"Warning: Using first finite solution as initial best (Fitness: {best_fitness:.4e})")
             else:
                 print("ERROR: No finite fitness solutions found in initial population.")
                 return self.initial_B_net.reshape(-1, 1), [], np.dot(self.A, self.initial_B_net)

        print(f"Initial Best Fitness: {best_fitness:.4e}")
        fitness_history = [best_fitness]
        algorithm_names = ['OOA', 'KHA', 'SHO']
        algo_success_count = {name: 0 for name in algorithm_names}; algo_attempts_count = {name: 0 for name in algorithm_names}
        algo_probabilities = {name: 1.0/len(algorithm_names) for name in algorithm_names}; adaptation_rate = 0.05
        stagnation_counter = 0; last_best_fitness = best_fitness

        for iteration in range(iterations):
            new_population = []; current_fitness_values = []
            # --- Adaptive Hybridization Logic (same as before) ---
            total_attempts = sum(algo_attempts_count.values())
            if total_attempts > 0:
                 success_rates = {name: algo_success_count[name] / algo_attempts_count[name] if algo_attempts_count[name] > 0 else 0 for name in algorithm_names}
                 total_rate = sum(success_rates.values())
                 if total_rate > 1e-6:
                      target_probabilities = {name: rate / total_rate for name, rate in success_rates.items()}
                      for name in algorithm_names: algo_probabilities[name] = (1 - adaptation_rate) * algo_probabilities[name] + adaptation_rate * target_probabilities[name]
                      prob_sum = sum(algo_probabilities.values())
                      if prob_sum > 1e-6: algo_probabilities = {name: p / prob_sum for name, p in algo_probabilities.items()}
                      else: algo_probabilities = {name: 1.0/len(algorithm_names) for name in algorithm_names}
                 current_weights = [algo_probabilities[name] for name in algorithm_names]
            else: current_weights = [1.0/len(algorithm_names)] * len(algorithm_names)
            # --- --- ---

            for i in range(population_size):
                try: algorithm = random.choices(algorithm_names, weights=current_weights, k=1)[0]
                except ValueError: algorithm = random.choice(algorithm_names)
                algo_attempts_count[algorithm] += 1
                current_solution_Bnet = initial_population[i]

                # Apply global search
                if algorithm == 'OOA': candidate_Bnet = self._apply_orcas_optimization(initial_population, i, best_solution_Bnet, iteration, iterations)
                elif algorithm == 'KHA': candidate_Bnet = self._apply_krill_herd(initial_population, i, best_solution_Bnet, iteration, iterations)
                else: candidate_Bnet = self._apply_spotted_hyena(initial_population, i, best_solution_Bnet, iteration, iterations)

                # Apply local search
                refined_candidate_Bnet = self._apply_local_search(candidate_Bnet, iteration, iterations)
                # Apply constraints
                new_solution_Bnet = self._apply_constraints(refined_candidate_Bnet)
                new_population.append(new_solution_Bnet)
                # Evaluate fitness
                new_fitness = self._calculate_fitness(new_solution_Bnet)
                current_fitness_values.append(new_fitness)

                # Update best
                if np.isfinite(new_fitness) and new_fitness < best_fitness:
                    best_fitness = new_fitness; best_solution_Bnet = new_solution_Bnet.copy()
                    algo_success_count[algorithm] += 1

            initial_population = new_population; fitness_values = current_fitness_values
            if np.isfinite(best_fitness): fitness_history.append(best_fitness)

            # --- Stagnation & Diversification Logic (same as before) ---
            if abs(best_fitness - last_best_fitness) < self.tolerance * max(1.0, abs(last_best_fitness)): stagnation_counter += 1
            else: stagnation_counter = 0; last_best_fitness = best_fitness
            if stagnation_counter >= self.stagnation_threshold:
                print(f"\nStagnation detected at iteration {iteration}. Diversifying population...")
                num_to_replace = int(population_size * self.diversification_fraction)
                worst_indices = np.argsort(fitness_values)[-num_to_replace:]
                for idx in worst_indices:
                    initial_population[idx] = self._generate_random_solution()
                    fitness_values[idx] = self._calculate_fitness(initial_population[idx])
                best_idx = np.argmin(fitness_values); best_solution_Bnet = initial_population[best_idx].copy()
                best_fitness = fitness_values[best_idx]; last_best_fitness = best_fitness; stagnation_counter = 0
            # --- --- ---

            if iteration % 100 == 0 or iteration == iterations - 1:
                deviation = self._get_generator_deviation(best_solution_Bnet); cost = self._get_rescheduling_cost(best_solution_Bnet)
                print(f"Iter {iteration}/{iterations}: BestFit={best_fitness:.4e}, GenDev(Bnet)={deviation:.3f}, GenCost(Bnet)={cost:.2f} (Stag: {stagnation_counter}/{self.stagnation_threshold})")

        best_solution_Bnet = self._apply_constraints(best_solution_Bnet) # Final constraint check
        # --- Final Reporting (same as before) ---
        total_attempts_final = sum(algo_attempts_count.values())
        if total_attempts_final > 0: print("\nAlgorithm Contributions (Attempts):", {k: f"{v/total_attempts_final*100:.1f}%" for k, v in algo_attempts_count.items()})
        final_deviation = self._get_generator_deviation(best_solution_Bnet); final_gen_cost = self._get_rescheduling_cost(best_solution_Bnet)
        print(f"\nFinal Sum of Absolute Changes (Generators Only - Bnet): {final_deviation:.4f}")
        print(f"Final Generator Rescheduling Cost (Based on Bnet change): {final_gen_cost:.2f} $/hr")
        # --- --- ---

        final_C = np.dot(self.A, best_solution_Bnet) # Flows from final Bnet
        return best_solution_Bnet, fitness_history, final_C # Return final Bnet

    # --- Memory Functions (Optional, operate on B_net) ---
    def _check_memory(self, B_unused):
        if not self.memory: return None
        for _, stored_Bnet in reversed(self.memory):
            if self._is_feasible(stored_Bnet): print("Using feasible B_net solution from memory."); return stored_Bnet
        return None
    def _store_in_memory(self, initial_Bnet_state, optimized_Bnet_state):
        self.memory.append((initial_Bnet_state.copy(), optimized_Bnet_state.copy()))
        print(f"Feasible B_net solution stored. Memory size: {len(self.memory)}")

    # --- Metaheuristic Implementations (OOA, KHA, SHO - operate on B_net) ---
    # These methods take B_net vectors as input and return a new B_net vector
    def _apply_orcas_optimization(self,p,i,b,it,m_it):
        s=p[i].copy();b_f=b.flatten();s_f=s.flatten();a=2*(1-(it/m_it)**2);r1,r2=random.random(),random.random();
        if r1<0.5: d=np.abs(b_f-s_f);l=2*r2-1;d=np.maximum(d,1e-9);step=a*r2*(b_f-s_f);n_f=s_f+step
        else: idx=[j for j in range(len(p)) if j!=i];X=p[random.choice(idx)].flatten() if idx else s_f;A=2*a*r1-a;C=2*r2;D=np.abs(C*X-s_f);n_f=X-A*D
        n_f=np.nan_to_num(n_f,nan=np.mean(s_f),posinf=np.max(s_f)*2,neginf=np.min(s_f)*2); return n_f.reshape(-1,1)
    def _apply_krill_herd(self,p,i,b,it,m_it):
        s=p[i].copy();b_f=b.flatten();s_f=s.flatten();Dmax=0.005*(1-it/m_it);Vf=.02;Nmax=.01;Dt=1.0;
        Ni=Nmax*(b_f-s_f);Fi=Vf*(b_f-s_f);d=np.random.uniform(-1,1,s_f.shape);Di=Dmax*d; n_f=s_f+Dt*(Ni+Fi+Di);
        n_f=np.nan_to_num(n_f,nan=np.mean(s_f),posinf=np.max(s_f)*2,neginf=np.min(s_f)*2); return n_f.reshape(-1,1)
    def _apply_spotted_hyena(self,p,i,b,it,m_it):
        s=p[i].copy();b_f=b.flatten();s_f=s.flatten();h=5-it*(5/m_it);B=2*random.random();E=2*h*random.random()-h;
        D_b=np.abs(B*b_f-s_f);X1=b_f-E*D_b;
        if abs(E)>=1: idx=[j for j in range(len(p)) if j!=i];r_h=p[random.choice(idx)].flatten() if idx else s_f;D_h=np.abs(B*r_h-s_f);n_f=r_h-E*D_h
        else: n_f=X1
        n_f=np.nan_to_num(n_f,nan=np.mean(s_f),posinf=np.max(s_f)*2,neginf=np.min(s_f)*2); return n_f.reshape(-1,1)



In [3]:
# Cell 3: Wrapper Function (optimize_power_flow_free_loadshed - Net Injection Model)

# Added fixed_load_full argument
def optimize_power_flow_free_loadshed(A, initial_B_net, fixed_load_full, # Pass initial B_net, fixed load
                                      line_limits, gen_costs_full, gen_limits_min_full, gen_limits_max_full,
                                      gen_indices, load_indices,
                                      iterations=5000, population_size=100):
    """
    Wrapper function for Net Injection model.
    Handles fixed loads and controllable generation.
    Objective: 1. Constraints & Min Gen Deviation (in Bnet), 2. Min Gen Cost (from Bnet change).
    Allows load shedding ONLY for buses in load_indices.

    Args:
        A (np.ndarray): System matrix.
        initial_B_net (np.ndarray): Initial NET bus injections (n_buses x 1).
        fixed_load_full (np.ndarray): Fixed load demand (Pl >= 0) for ALL buses (n_buses x 1 or flat).
        line_limits (np.ndarray): Absolute limits for line flows (n_lines).
        gen_costs_full (np.ndarray): Cost coefficients for ALL buses (n_buses).
        gen_limits_min_full (np.ndarray): Min generation (Pg) limit for ALL buses (n_buses).
        gen_limits_max_full (np.ndarray): Max generation (Pg) limit for ALL buses (n_buses).
        gen_indices (list or np.ndarray): 0-based indices for generator buses.
        load_indices (list or np.ndarray): 0-based indices for load-only buses.
        iterations (int): Number of optimization iterations.
        population_size (int): Number of solutions in the population.

    Returns:
        tuple: (B_optimized_net, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details)
               - B_optimized_net: Optimized NET bus injections.
               - ... (rest are same as before)
    """
    print("--- Initializing Optimizer (Net Injection Model, Load Shed Allowed for Loads) ---")
    try:
        if 'HybridPowerFlowOptimizer' not in globals():
             raise NameError("Optimizer class 'HybridPowerFlowOptimizer' is not defined.")

        # Pass all required arguments, including fixed_load_full
        optimizer = HybridPowerFlowOptimizer(A, line_limits, gen_costs_full, gen_limits_min_full, gen_limits_max_full,
                                             initial_B_net, fixed_load_full, # <-- Pass B_net and Pl
                                             gen_indices, load_indices)

    except (ValueError, NameError) as e:
        print(f"ERROR initializing optimizer: {e}")
        try: C_unoptimized_calc = np.dot(A, initial_B_net) # Use initial B_net for C_unopt
        except Exception: C_unoptimized_calc = np.full((A.shape[0], 1), np.nan)
        return initial_B_net, [], C_unoptimized_calc, C_unoptimized_calc, False, {} # Return initial B_net

    # Get the potentially adjusted initial B_net from the optimizer
    initial_Bnet_from_opt = optimizer.initial_B_net.reshape(-1, 1)
    try: C_unoptimized = np.dot(A, initial_Bnet_from_opt)
    except Exception as e: C_unoptimized = np.full((A.shape[0], 1), np.nan)

    print("\n--- Checking Initial State Feasibility (Using Optimizer's Initial B_net) ---")
    initial_feasible = optimizer._is_feasible(initial_Bnet_from_opt, verbose=True)
    print(f"Optimizer's initial state feasible: {initial_feasible}")
    if not initial_feasible: print("WARNING: Optimizer starting from an infeasible state.")

    print("\n--- Starting Optimization (Net Injection Model) ---")
    # Run optimization - operates internally on B_net
    B_opt_net, fit_hist, C_opt = optimizer.optimize(None, iterations=iterations, population_size=population_size)

    print("\n--- Checking Final Solution Feasibility ---")
    final_feasible = optimizer._is_feasible(B_opt_net, verbose=True)
    print(f"\nFinal feasibility: {final_feasible}")

    details = {"gen_indices": optimizer.gen_indices, "load_indices": optimizer.load_indices,
               "fixed_load": optimizer.fixed_load} # Include fixed load in details

    # Return optimized B_net
    return B_opt_net, fit_hist, C_opt, C_unoptimized, final_feasible, details


In [4]:
# Cell 4: Visualization Function (visualize_results - Minor changes for robustness)

def visualize_results(A, B, B_optimized, C_optimized, C_unoptimized, line_limits,
                      gen_costs, gen_limits_min, gen_limits_max,
                      gen_indices, load_indices, fitness_history):
    """
    Visualizes the optimization results.
    Includes checks for valid data before plotting.
    """
    num_lines = A.shape[0]
    num_buses = A.shape[1]
    tolerance = 1e-6 # Tolerance for violation checks

    print("\n--- Generating Plots (Enhanced Hybrid - Load Shed Allowed - Zero Cost) ---")

    # Plot 1: Line Flows
    try:
        plt.figure(figsize=(12, 6))
        idx = np.arange(1, num_lines + 1)

        # Plot flows only if they are valid numpy arrays
        if isinstance(C_unoptimized, np.ndarray) and C_unoptimized.size == num_lines:
             plt.plot(idx, C_unoptimized.flatten(), 'o-', label='Unoptimized', alpha=0.7, markersize=4)
        else: print("Warning: Skipping unoptimized flows plot (invalid data).")

        if isinstance(C_optimized, np.ndarray) and C_optimized.size == num_lines:
             plt.plot(idx, C_optimized.flatten(), 's--', label='Optimized', alpha=0.9, markersize=5)
             # Check for violations in optimized flows
             flows_opt_abs = np.abs(C_optimized.flatten())
             violations = np.where(flows_opt_abs > line_limits + tolerance)[0]
             if len(violations) > 0:
                 plt.scatter(idx[violations], C_optimized.flatten()[violations], c='magenta', s=100, zorder=5, label=f'Violations ({len(violations)})', marker='x')
        else: print("Warning: Skipping optimized flows plot (invalid data).")

        # Plot limits
        plt.plot(idx, line_limits, 'r:', alpha=0.8, label='Limit (+)')
        plt.plot(idx, -line_limits, 'r:', alpha=0.8, label='Limit (-)')

        plt.xlabel('Line Index')
        plt.ylabel('Power Flow (MW or p.u.)')
        plt.title('Line Flows Comparison (Load Shed Allowed - Zero Cost)')
        plt.legend()
        plt.grid(True, linestyle=':')
        plt.xticks(idx[::max(1, num_lines//20)]) # Adjust x-ticks density
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Plotting error (Line Flows): {e}")

    # Plot 2: Fitness History
    try:
        # Check if fitness_history is a list/array with more than one point
        if isinstance(fitness_history, (list, np.ndarray)) and len(fitness_history) > 1:
            plt.figure(figsize=(10, 5))
            # Filter out non-finite values for plotting
            finite_fitness = [f for f in fitness_history if np.isfinite(f)]
            if len(finite_fitness) > 1:
                 plt.plot(finite_fitness, '.-', color='royalblue', label='Best Fitness', markersize=3, linewidth=1)
                 plt.xlabel('Iteration')
                 plt.ylabel('Fitness Value (Log Scale)')
                 plt.title('Optimization Convergence')
                 plt.yscale('log') # Use log scale for potentially large fitness values
                 plt.legend()
                 plt.grid(True, linestyle=':')
                 plt.tight_layout()
                 plt.show()
            else: print("Warning: Not enough finite fitness values to plot convergence.")
        else: print("Warning: Skipping fitness plot (insufficient history data).")
    except Exception as e:
        print(f"Plotting error (Fitness History): {e}")

    # Plot 3: Bus Injections
    try:
        plt.figure(figsize=(14, 7))
        bus_idx_plot = np.arange(1, num_buses + 1)
        bar_width = 0.35

        # Define colors based on generator/load status
        colors_initial = ['darkblue' if i in gen_indices else 'skyblue' for i in range(num_buses)]
        colors_optimized = ['darkgreen' if i in gen_indices else 'lightgreen' for i in range(num_buses)]

        # Plot initial and optimized injections
        if isinstance(B, np.ndarray) and B.size == num_buses:
             plt.bar(bus_idx_plot - bar_width/2, B.flatten(), width=bar_width, label='Initial (Gen=dark)', alpha=0.7, color=colors_initial)
        else: print("Warning: Skipping initial injections plot (invalid data).")

        if isinstance(B_optimized, np.ndarray) and B_optimized.size == num_buses:
             plt.bar(bus_idx_plot + bar_width/2, B_optimized.flatten(), width=bar_width, label='Optimized (Gen=dark)', alpha=0.8, color=colors_optimized)
        else: print("Warning: Skipping optimized injections plot (invalid data).")

        # Plot Generator Limits
        if len(gen_indices) > 0:
             # Ensure limits arrays are valid
             if isinstance(gen_limits_max, np.ndarray) and len(gen_limits_max) == num_buses and \
                isinstance(gen_limits_min, np.ndarray) and len(gen_limits_min) == num_buses:
                 plt.scatter(bus_idx_plot[gen_indices], gen_limits_max[gen_indices], c='dimgrey', marker='_', s=150, label='Gen Max Limit', zorder=5)
                 plt.scatter(bus_idx_plot[gen_indices], gen_limits_min[gen_indices], c='dimgrey', marker='_', s=150, label='Gen Min Limit', zorder=5)
             else: print("Warning: Skipping generator limit markers (invalid limit data).")

        # Plot Load Limits (Max = 0, Min = Initial Load)
        if len(load_indices) > 0:
             if isinstance(B, np.ndarray) and B.size == num_buses:
                 plt.scatter(bus_idx_plot[load_indices], np.zeros(len(load_indices)), c='lightcoral', marker='_', s=150, label='Load Max Limit (0)', zorder=5)
                 plt.scatter(bus_idx_plot[load_indices], B.flatten()[load_indices], c='lightcoral', marker='_', s=150, label='Load Min Limit (Initial)', zorder=5)
             else: print("Warning: Skipping load limit markers (invalid initial B data).")


        plt.xlabel('Bus Index')
        plt.ylabel('Power Injection (MW or p.u.)')
        plt.title('Bus Power Injections: Initial vs. Optimized')
        plt.xticks(bus_idx_plot[::max(1, num_buses//20)]) # Adjust x-ticks density
        plt.legend()
        plt.grid(True, axis='y', linestyle=':')
        plt.axhline(0, color='black', linewidth=0.5) # Zero line
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Plotting error (Bus Injections): {e}")

    # Plot 4 & 5: Changes in Generation and Load
    try:
        # Ensure B and B_optimized are valid before calculating changes
        if isinstance(B, np.ndarray) and B.size == num_buses and \
           isinstance(B_optimized, np.ndarray) and B_optimized.size == num_buses:
            changes = B_optimized.flatten() - B.flatten()
            bus_idx_plot = np.arange(1, num_buses + 1) # Recalculate just in case

            # Generator Changes Plot
            if len(gen_indices) > 0:
                plt.figure(figsize=(10, 4))
                gen_changes = changes[gen_indices]
                gen_labels = bus_idx_plot[gen_indices]
                colors_gen = ['forestgreen' if x >= 0 else 'firebrick' for x in gen_changes]
                bar_indices_gen = np.arange(len(gen_indices))
                plt.bar(bar_indices_gen, gen_changes, color=colors_gen)
                plt.axhline(0, color='black', linestyle='-', linewidth=0.7)
                plt.xlabel('Generator Bus Index')
                plt.ylabel('Change in Injection')
                plt.title('Generator Output Changes (Optimized - Initial)')
                plt.xticks(bar_indices_gen, gen_labels)
                plt.grid(True, axis='y', linestyle=':')
                plt.tight_layout()
                plt.show()

            # Load Changes (Shedding) Plot
            if len(load_indices) > 0:
                plt.figure(figsize=(10, 4))
                # Positive change for load means injection increased (less negative) -> load shed
                load_changes = changes[load_indices]
                load_shed = np.maximum(0, load_changes) # Only show shedding (increase towards 0)
                load_labels = bus_idx_plot[load_indices]
                # Color bars where shedding occurred
                colors_load = ['darkorange' if x > tolerance else 'darkgrey' for x in load_shed]
                bar_indices_load = np.arange(len(load_indices))
                plt.bar(bar_indices_load, load_shed, color=colors_load)
                plt.axhline(0, color='black', linestyle='-', linewidth=0.7)
                plt.xlabel('Load Bus Index')
                plt.ylabel('Load Shed Amount (MW or p.u.)')
                plt.title('Load Shedding (Optimized Injection - Initial Injection)')
                plt.xticks(bar_indices_load, load_labels)
                plt.grid(True, axis='y', linestyle=':')
                plt.ylim(bottom= -0.05 * max(1, np.max(load_shed)) if np.any(load_shed > 0) else -0.1) # Adjust bottom limit slightly
                plt.tight_layout()
                plt.show()
        else:
             print("Warning: Skipping change plots (invalid B or B_optimized data).")

    except Exception as e:
        print(f"Plotting error (Changes): {e}")

    # --- Text Summary ---
    print("\n" + "="*30 + " RESULTS SUMMARY (Enhanced Hybrid) " + "="*30)
    np.set_printoptions(precision=4, suppress=True) # Adjust precision for summary
    try:
        # Calculate final metrics only if indices are valid
        final_dev_g = 0.0
        final_g_cost = 0.0
        total_ls = 0.0

        if len(gen_indices) > 0 and isinstance(B_optimized, np.ndarray) and B_optimized.size==num_buses \
           and isinstance(B, np.ndarray) and B.size==num_buses \
           and isinstance(gen_costs, np.ndarray) and gen_costs.size==num_buses:
             g_costs_only = gen_costs.flatten()[gen_indices]
             gen_deviation_vector = np.abs(B_optimized.flatten()[gen_indices] - B.flatten()[gen_indices])
             final_dev_g = np.sum(gen_deviation_vector)
             final_g_cost = np.sum(g_costs_only * gen_deviation_vector)
        elif len(gen_indices) > 0:
             print("Warning: Could not calculate final generator metrics due to invalid data/indices.")

        if len(load_indices) > 0 and isinstance(B_optimized, np.ndarray) and B_optimized.size==num_buses \
           and isinstance(B, np.ndarray) and B.size==num_buses:
             load_shed_amount = np.maximum(0, B_optimized.flatten()[load_indices] - B.flatten()[load_indices])
             total_ls = np.sum(load_shed_amount)
        elif len(load_indices) > 0:
             print("Warning: Could not calculate final load shed due to invalid data/indices.")


        print(f"\nInitial B:\n{B.flatten() if isinstance(B, np.ndarray) else 'N/A'}")
        print(f"\nOptimized B:\n{B_optimized.flatten() if isinstance(B_optimized, np.ndarray) else 'N/A'}")
        if isinstance(B_optimized, np.ndarray): print(f"Sum Optimized B: {np.sum(B_optimized):.6f}")

        print(f"\nInitial Flows:\n{C_unoptimized.flatten() if isinstance(C_unoptimized, np.ndarray) else 'N/A'}")
        print(f"\nOptimized Flows:\n{C_optimized.flatten() if isinstance(C_optimized, np.ndarray) else 'N/A'}")
        print("-" * 70)
        print("\nObjective Metrics & Load Shed:")
        print(f"  Generator Deviation Sum: {final_dev_g:.4f}")
        print(f"  Generator Rescheduling Cost: {final_g_cost:.2f}")
        print(f"  Total Load Shed: {total_ls:.4f} MW (or p.u.)")

        print("\nConstraint Check Summary:")
        # Re-check feasibility using the class method for consistency
        try:
            # Need to instantiate the optimizer again just for the check, which isn't ideal
            # Or, pass the optimizer object to visualize_results if feasible
            # Simple re-check here:
            temp_opt = HybridPowerFlowOptimizer(A, line_limits, gen_costs, gen_limits_min, gen_limits_max, B)
            is_final_feasible = temp_opt._is_feasible(B_optimized) # Use the final optimized B
            print(f"  Final solution feasible (re-checked): {'YES' if is_final_feasible else 'NO'}")
        except Exception as e:
            print(f"Error during final feasibility re-check: {e}")
            print("  Final solution feasibility: Unknown (check failed)")

    except Exception as e:
        print(f"Error generating summary text: {e}")

    finally:
        np.set_printoptions(precision=8, suppress=False) # Reset numpy print options

    print("=" * 70)
    print("Note: Visualization assumes MW or consistent p.u. units. Voltage constraints not included.")
    print("=" * 70)


In [ ]:
# Cell 5: Main Execution Block (Net Injection Model - Syntax Fixed)
import numpy as np
import pandas as pd
import sys
import time

# --- Ensure functions/classes from previous cells are available ---
# In a real notebook, ensure Cells 1-4 have been executed
if 'HybridPowerFlowOptimizer' not in globals(): print("ERROR: Cell 2 (HybridPowerFlowOptimizer class) not executed."); exit()
if 'optimize_power_flow_free_loadshed' not in globals(): print("ERROR: Cell 3 (optimize_power_flow_free_loadshed function) not executed."); exit()
if 'visualize_results' not in globals(): print("ERROR: Cell 4 (visualize_results function) not executed."); exit()


if __name__ == "__main__":

    matrix_file_name = 'reshaped_data.csv'
    try:
        print(f"Loading system matrix A from: {matrix_file_name}")
        df = pd.read_csv(matrix_file_name, header=None); A = df.to_numpy()
        print(f"Successfully loaded matrix A with shape {A.shape}")
        if A.ndim != 2 or A.shape[0] == 0 or A.shape[1] == 0: raise ValueError("Invalid matrix A.")
    except Exception as e: print(f"FATAL ERROR loading matrix A: {e}. Exiting."); exit()

    num_lines_main = A.shape[0]; num_buses_main = A.shape[1]
    print(f"System dimensions: {num_lines_main} lines, {num_buses_main} buses.")
    all_bus_indices_set = set(range(num_buses_main)) # 0-based indices

    # --- Main Scenario Loop ---
    while True:
        print("\n" + "="*25 + " New Scenario (Net Injection Model) " + "="*25)
        print("Objective: 1. Meet Limits & Min Gen Deviation, 2. Min Gen Cost")
        print("(Load Shedding Allowed for Load-Only Buses)")

        try:
            # --- Get User Inputs ---

            # 1. Get Generator Indices (1-based)
            while True:
                 try:
                     gen_indices_str = input(f"\n>>> Enter GENERATOR bus numbers (1 to {num_buses_main}, space-separated): ")
                     user_gen_indices_list = [int(x) for x in gen_indices_str.split()]
                     valid_indices = True; temp_gen_indices_0based = []; gen_indices_set = set()
                     for idx_1based in user_gen_indices_list:
                         if 1 <= idx_1based <= num_buses_main:
                             idx_0based = idx_1based - 1
                             if idx_0based in gen_indices_set: print(f"  Error: Duplicate index {idx_1based}."); valid_indices = False; break
                             gen_indices_set.add(idx_0based); temp_gen_indices_0based.append(idx_0based)
                         else: print(f"  Error: Invalid bus number {idx_1based}."); valid_indices = False; break
                     if valid_indices: temp_gen_indices_0based.sort(); break
                 except ValueError: print("  Error: Invalid input format.")
                 except EOFError: raise

            # Determine load-only indices
            temp_load_indices_0based = sorted(list(all_bus_indices_set - gen_indices_set))
            print(f"-> Using Generators: {np.array(temp_gen_indices_0based) + 1}")
            print(f"-> Using Load-Only Buses: {np.array(temp_load_indices_0based) + 1}")

            # 2. Get Fixed Load (Pl) for ALL buses
            print(f"\n>>> Enter FIXED LOAD demand (Pl >= 0) for ALL buses ({num_buses_main} values):")
            print("    (Enter 0 for buses with no load)")
            while True:
                 try:
                     pl_str = input("  Fixed Loads (Pl): ")
                     pl_list = [float(x) for x in pl_str.split()]
                     if len(pl_list) == num_buses_main:
                         fixed_load_input = np.array(pl_list)
                         if np.any(fixed_load_input < 0):
                              print("  Warning: Negative loads entered. Treating them as zero load.")
                              fixed_load_input = np.maximum(0, fixed_load_input) # Ensure Pl >= 0
                         break
                     else: print(f"  Error: Expected {num_buses_main} values, got {len(pl_list)}.")
                 except ValueError: print("  Error: Invalid number format.")
                 except EOFError: raise

            # 3. Get Initial Generation (Pg_initial) ONLY for GENERATOR buses
            pg_initial_input = np.zeros(num_buses_main) # Initialize Pg for all buses to 0
            if len(temp_gen_indices_0based) > 0:
                print(f"\n>>> Enter INITIAL GENERATION (Pg) ONLY for GENERATOR buses {np.array(temp_gen_indices_0based) + 1}:")
                for i in temp_gen_indices_0based:
                    while True:
                        try:
                             pg_initial_input[i] = float(input(f"    G{i+1} Initial Pg: "))
                             break
                        except ValueError:
                             print("    Invalid number.")
                        except EOFError:
                             raise # Correct syntax
            else: print("\nNOTE: No generators specified, assuming initial Pg = 0 for all.")

            # 4. Calculate Initial NET Injection (B = Pg - Pl)
            B_input_net = (pg_initial_input - fixed_load_input).reshape(-1, 1)
            print("\nCalculated Initial Net Injections (B = Pg - Pl):")
            for i in range(num_buses_main): print(f"  Bus {i+1}: {B_input_net[i,0]:.4f}")

            # 5. Get Line Limits
            print(f"\n>>> Enter LINE power limits ({num_lines_main} values):")
            while True:
                 try:
                     limits_str = input("  Line limits: ")
                     line_limits_list = [float(x) for x in limits_str.split()]
                     if len(line_limits_list) == num_lines_main:
                         line_limits_input = np.abs(np.array(line_limits_list)); break
                     else: print(f"  Error: Expected {num_lines_main} values, got {len(line_limits_list)}.")
                 except ValueError: print("  Error: Invalid number format.")
                 except EOFError: raise

            # 6. Get Generator-Specific Data (Costs, Pg Limits)
            gen_costs_full_input = np.zeros(num_buses_main)
            gen_limits_pg_min_input = np.zeros(num_buses_main)
            gen_limits_pg_max_input = np.zeros(num_buses_main)
            if len(temp_gen_indices_0based) > 0:
                print(f"\n>>> Enter Cost/Limits ONLY for GENERATOR buses {np.array(temp_gen_indices_0based) + 1}:")
                print("  Enter Gen Cost Coefficients ($/MWh change):")
                for i in temp_gen_indices_0based:
                    while True:
                        try:
                             gen_costs_full_input[i] = float(input(f"    G{i+1} cost: "))
                             break
                        except ValueError:
                             print("    Invalid number.")
                        except EOFError:
                             raise # Correct syntax
                print("  Enter Gen MIN Generation Limit (Pg_min MW):")
                for i in temp_gen_indices_0based:
                    while True:
                        try:
                             gen_limits_pg_min_input[i] = float(input(f"    G{i+1} Pg_min: "))
                             break
                        except ValueError:
                             print("    Invalid number.")
                        except EOFError:
                             raise # Correct syntax
                print("  Enter Gen MAX Generation Limit (Pg_max MW):")
                for i in temp_gen_indices_0based:
                    while True:
                        try:
                            max_val = float(input(f"    G{i+1} Pg_max: "))
                            if max_val < gen_limits_pg_min_input[i]:
                                 print(f"    Error: Max Pg ({max_val}) < Min Pg ({gen_limits_pg_min_input[i]}). Re-enter Max.")
                                 continue
                            gen_limits_pg_max_input[i] = max_val
                            break
                        except ValueError:
                             print("    Invalid number.")
                        except EOFError:
                             raise # Correct syntax
            else: print("\nNOTE: No generators specified, skipping generator cost/limit input.")

        except EOFError: print("\nInput interrupted during setup. Restarting scenario..."); continue
        except Exception as e: print(f"\nAn unexpected error occurred during input: {e}. Restarting..."); continue

        # --- Call Optimization ---
        print("\n" + "="*25 + " Running Optimization (Net Injection Model) " + "="*25)
        try:
            opt_iterations = 5000; opt_pop_size = 100
            start_time = time.time()
            # Pass initial B_net, fixed_load, Pg limits, etc.
            B_optimized_net, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details = optimize_power_flow_free_loadshed(
                A, B_input_net, fixed_load_input, # Pass B_net and Pl
                line_limits_input, gen_costs_full_input, gen_limits_pg_min_input, gen_limits_pg_max_input, # Pass full arrays
                temp_gen_indices_0based, temp_load_indices_0based, # Pass indices
                iterations=opt_iterations,
                population_size=opt_pop_size
            )
            end_time = time.time()
            print(f"Optimization Duration: {end_time - start_time:.2f} seconds")
        except NameError as e: print(f"FATAL ERROR: Required function not defined ({e})."); break
        except Exception as e:
             print(f"An unexpected error occurred during optimization run: {e}")
             try:
                 if input("Try another scenario anyway? (y/n):").strip().lower() != 'y': break
                 else: continue
             except EOFError: print("\nInput interrupted. Exiting..."); break

        print("\n" + "="*25 + " Optimization Finished " + "="*25)

        # --- Post-processing: Calculate Optimized Pg and Load Shed ---
        final_fixed_load = opt_details.get("fixed_load", np.zeros(num_buses_main))
        optimized_Pg = np.zeros(num_buses_main)
        if len(opt_details["gen_indices"]) > 0:
             optimized_Pg[opt_details["gen_indices"]] = B_optimized_net.flatten()[opt_details["gen_indices"]] + final_fixed_load[opt_details["gen_indices"]]
        load_shed_vector = np.zeros(num_buses_main)
        if len(opt_details["load_indices"]) > 0:
             load_shed_vector[opt_details["load_indices"]] = final_fixed_load[opt_details["load_indices"]] + B_optimized_net.flatten()[opt_details["load_indices"]]
             load_shed_vector = np.maximum(0, load_shed_vector)

        # --- Visualize Results ---
        try:
            viz_gen_indices = opt_details.get("gen_indices", np.array([]))
            viz_load_indices = opt_details.get("load_indices", np.array([]))
            print("\nVisualizing results...")
            visualize_results(A, B_input_net, B_optimized_net, C_optimized, C_unoptimized, line_limits_input,
                              gen_costs_full_input, gen_limits_pg_min_input, gen_limits_pg_max_input, # Pass Pg limits
                              viz_gen_indices, viz_load_indices,
                              fitness_history)
        except NameError as e: print(f"Error during visualization: Required function not defined ({e}).")
        except Exception as e: print(f"An error occurred during visualization: {e}")

        # --- Save Results ---
        if final_feasible:
            try:
                save = input("\nSave detailed results to file? (y/n): ").strip().lower()
                if save == 'y':
                    default_fname = "power_flow_net_injection_results.txt"
                    fname = input(f"Enter filename (default: {default_fname}): ").strip() or default_fname
                    print(f"Attempting to save results to {fname}...")
                    try:
                        with open(fname, 'w') as f:
                            save_gen_indices = opt_details.get("gen_indices", np.array([]))
                            save_load_indices = opt_details.get("load_indices", np.array([]))
                            gen_dev = 0.0; gen_cost = 0.0; total_load_shed = np.sum(load_shed_vector)
                            gen_costs_save = []
                            if len(save_gen_indices) > 0:
                                gen_costs_save=gen_costs_full_input[save_gen_indices]
                                dev_vec = np.abs(B_optimized_net.flatten()[save_gen_indices] - B_input_net.flatten()[save_gen_indices])
                                gen_dev = np.sum(dev_vec); gen_cost = np.sum(gen_costs_save * dev_vec)

                            f.write("Power Flow Opt Results (Net Injection Model, Load Shed Allowed for Loads)\n")
                            f.write("Objective: 1. Limits & Min Gen Deviation, 2. Min Gen Cost\n"); f.write("="*30+"\n\n")
                            f.write(f"Matrix A (Shape: {A.shape}):\n"); np.savetxt(f, A, fmt='%.4f'); f.write("\n")
                            f.write("Line Limits:\n"); [f.write(f"L{i+1}: {l:.2f}\n") for i,l in enumerate(line_limits_input)]
                            f.write("\nFixed Loads (Pl):\n"); [f.write(f"Bus {i+1}: {pl:.4f}\n") for i,pl in enumerate(final_fixed_load)]
                            f.write("\nGen Costs:\n"); [f.write(f"G{save_gen_indices[i]+1}: {c:.2f}\n") for i,c in enumerate(gen_costs_save)]
                            f.write("\nGen Min Generation (Pg_min):\n"); [f.write(f"G{idx+1}: {gen_limits_pg_min_input[idx]:.2f}\n") for idx in save_gen_indices]
                            f.write("\nGen Max Generation (Pg_max):\n"); [f.write(f"G{idx+1}: {gen_limits_pg_max_input[idx]:.2f}\n") for idx in save_gen_indices]
                            f.write("\nInitial Net Injection (B_net = Pg_initial - Pl):\n"); [f.write(f"Bus {i+1}: {b[0]:.4f}\n") for i,b in enumerate(B_input_net)]
                            f.write("\nOptimized Net Injection (B_net):\n"); [f.write(f"Bus {i+1}: {b[0]:.4f}\n") for i,b in enumerate(B_optimized_net)]
                            f.write("\nOptimized Generation (Pg = B_net_opt + Pl at Gen buses):\n"); [f.write(f"Bus {idx+1}: {optimized_Pg[idx]:.4f}\n") for idx in save_gen_indices]
                            f.write("\nLoad Shed Amount (at Load-Only buses):\n"); [f.write(f"Bus {idx+1}: {load_shed_vector[idx]:.4f}\n") for idx in save_load_indices if load_shed_vector[idx] > 1e-6] # Only show where shed > 0
                            f.write("\nInitial Flows (C_unopt = A * B_net_initial):\n"); [f.write(f"L{i+1}: {c[0]:.4f}\n") for i,c in enumerate(C_unoptimized)]
                            f.write("\nOptimized Flows (C_opt = A * B_net_opt):\n"); [f.write(f"L{i+1}: {c[0]:.4f}\n") for i,c in enumerate(C_optimized)]
                            f.write(f"\nFinal Gen Deviation Sum (B_net based): {gen_dev:.4f}\n")
                            f.write(f"Final Gen Rescheduling Cost (B_net based): {gen_cost:.2f}\n")
                            f.write(f"Total Load Shed (at Load-Only buses): {total_load_shed:.4f} MW (or p.u.)\n")
                            f.write(f"\nFeasible: {'YES' if final_feasible else 'NO'}\n")
                        print(f"Results successfully saved to {fname}")
                    except IOError as e: print(f"ERROR saving results to file '{fname}': {e}")
                    except IndexError as e: print(f"ERROR saving results: Index out of bounds - {e}.")
                    except Exception as e: print(f"An unexpected error occurred during saving: {e}")
            except EOFError:
                print("\nInput interrupted during save prompt.")
                if input("Continue to next scenario anyway? (y/n):").strip().lower() != 'y': break
        elif not final_feasible: print("\nFinal solution was infeasible. Results not saved.")

        # --- Ask to run again ---
        try:
            run_again = input("\nRun another scenario? (y/n):").strip().lower()
            if run_again != 'y': print("\nExiting..."); break
            else: print("\nRestarting scenario...\n" + "-"*70)
        except EOFError: print("\nInput interrupted. Exiting..."); break

    print("\nScript finished.")


Loading system matrix A from: reshaped_data.csv
Successfully loaded matrix A with shape (41, 30)
System dimensions: 41 lines, 30 buses.

========================= New Scenario (Net Injection Model) =========================
Objective: 1. Meet Limits & Min Gen Deviation, 2. Min Gen Cost
(Load Shedding Allowed for Load-Only Buses)



>>> Enter GENERATOR bus numbers (1 to 30, space-separated):  1 2 5 8 11 13


-> Using Generators: [ 1  2  5  8 11 13]
-> Using Load-Only Buses: [ 3  4  6  7  9 10 12 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30]

>>> Enter FIXED LOAD demand (Pl >= 0) for ALL buses (30 values):
    (Enter 0 for buses with no load)


  Fixed Loads (Pl):  0 21.7 2.40 7.60 94.2 0 22.8 30 0 5.8 0 11.2 0 6.2 8.2 3.5 9 3.2 9.5 2.2 17


  Error: Expected 30 values, got 21.


  Fixed Loads (Pl):  0 21.7 2.40 7.60 94.2 0 22.8 30 0 5.8 0 11.2 0 6.2 8.2 3.5 9 3.2 9.5 2.2 17.5 0 3.2 8.7 0 3.5 0 0 2.4 10.6



>>> Enter INITIAL GENERATION (Pg) ONLY for GENERATOR buses [ 1  2  5  8 11 13]:
